Import the required modules and functions

In [ ]:
from bs4 import BeautifulSoup
import requests
import os

Download the homepage and find the number of pages in the website

In [ ]:
homepage_url = 'https://windows10spotlight.com/'

homepage_url_data = requests.get(homepage_url).text

homepage_soup = BeautifulSoup(homepage_url_data,"html.parser")

title = homepage_soup.find(id='productTitle').get_text().strip()

number_of_pages = int(homepage_soup.find_all('a')[20].text)

Download each page in the website to collect the image id's in each of it

In [ ]:
def clean_caption(caption):
    return caption.replace('/',' ').replace('\\',' ')

def download_image_pc(link, caption, extension):

    file_name = caption + '.' + extension
    
    link_data = requests.get(link)

    os.chdir(r'P:\Codes\Projects\Python\Bing Image Downloader\Bing Images\PC Version')
    path=os.path.join(os.getcwd() , file_name)
    
    with open(path, 'wb') as f:
        f.write(link_data.content)

def download_image_mobile(link, caption, extension):

    file_name = caption + '.' + extension
    
    link_data = requests.get(link)

    os.chdir(r'P:\Codes\Projects\Python\Bing Image Downloader\Bing Images\Mobile Version')
    path=os.path.join(os.getcwd() , file_name)
    
    with open(path, 'wb') as f:
        f.write(link_data.content)
        print(file_name)
    
def redirect_to_image(image_id):

    image_page_url = 'https://windows10spotlight.com/images/' + str(image_id)

    image_page_url_data = requests.get(image_page_url).text
    
    image_page_soup = BeautifulSoup(image_page_url_data, 'html.parser')

    #PC version
    for link in image_page_soup.find_all('figure'):
        image_caption = clean_caption(link.figcaption.text)

        #pc version
        pc_image_link = link.a.get('href')
        pc_image_extension = pc_image_link.split('.')[-1]
        download_image_pc(pc_image_link, image_caption, pc_image_extension)

        #mobile version
        parent_link = link.parent
        mobile_image_link = parent_link.findAll('p')[1].a.get('href')
        mobile_image_extension = mobile_image_link.split('.')[-1]
        download_image_mobile(mobile_image_link, image_caption, mobile_image_extension)


def redirect_to_page(page_number):

    page_url = 'https://windows10spotlight.com/page/' + str(page_number)

    page_url_data = requests.get(page_url).text

    page_soup = BeautifulSoup(page_url_data, 'html.parser')

    for link in page_soup.find_all('a', href = True):
        if(link.img != None):
            img_id = link.img['alt']
            redirect_to_image(img_id)
       

for page_number in range(1,number_of_pages+1):

    redirect_to_page(page_number)

